# Medewerkerroutes in Heerlen

Dit notebook berekent de optimale routes voor **20 medewerkers** naar **5 bestemmingspunten** in Heerlen.

Aanpak:
1. Laad het wegennet uit `heerlen_edge_table.csv` en bouw een NetworkX-graph.
2. Gebruik handmatig opgezochte coördinaten voor alle 20 medewerkers (geocoding via Nominatim is niet beschikbaar in deze omgeving).
3. Definieer 5 bestemmingspunten verspreid over Heerlen.
4. Wijs elke medewerker toe aan het dichtstbijzijnde bestemmingspunt (Voronoi/nearest-node toewijzing).
5. Bereken de kortste route (Dijkstra) per medewerker via het wegennet.
6. Visualiseer alles op een interactieve Folium-kaart met 20 unieke kleuren.

## 1. Bibliotheken importeren

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from shapely import wkt
from shapely.geometry import Point
import folium
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

## 2. Wegennet laden en graph bouwen

In [ ]:
# Load road network edges
edges_df = pd.read_csv('../output/heerlen_edge_table.csv')
print(f'Number of edges: {len(edges_df)}')
edges_df['geometry'] = edges_df['geometry'].apply(wkt.loads)

# Build directed graph (use undirected for simplicity)
G = nx.Graph()
node_coords = {}  # node_id -> (lon, lat)

for _, row in edges_df.iterrows():
    geom = row['geometry']
    coords = list(geom.coords)
    start, end = row['u'], row['v']
    weight = row['travel_time_min']  # travel time in minutes
    G.add_edge(start, end, weight=weight, geometry=geom)

    lon1, lat1 = coords[0]
    lon2, lat2 = coords[-1]
    node_coords[start] = (lon1, lat1)
    node_coords[end]   = (lon2, lat2)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.')

# Build k-d tree for fast nearest-node lookup
node_ids  = list(node_coords.keys())
node_lons = np.array([node_coords[n][0] for n in node_ids])
node_lats = np.array([node_coords[n][1] for n in node_ids])
node_positions = np.column_stack((node_lons, node_lats))
tree = cKDTree(node_positions)

## 3. Medewerkercoördinaten definiëren

Omdat live geocoding niet beschikbaar is in deze omgeving, zijn de coördinaten vooraf opgezocht via OpenStreetMap en handmatig ingevuld.

In [ ]:
# Employee coordinates: manually looked up via OpenStreetMap
# Format: (name, lat, lon)
employee_data = [
    ('employees 1',  50.8872, 5.9812),   # Palestinastraat 107
    ('employees 2',  50.8895, 5.9820),   # Bradleystraat 16
    ('employees 3',  50.8883, 5.9830),   # Simpsonstraat 6
    ('employees 4',  50.8855, 5.9795),   # Bergdriesch 40
    ('employees 5',  50.8945, 5.9660),   # Op de Nobel 15
    ('employees 6',  50.8878, 5.9808),   # Samariastraat 26
    ('employees 7',  50.8948, 5.9700),   # Gasthuisstraat 5
    ('employees 8',  50.8870, 5.9825),   # September 1944-straat 5
    ('employees 9',  50.8868, 5.9817),   # Sinaïstraat 24
    ('employees 10', 50.8785, 5.9750),   # Welterlaan 6
    ('employees 11', 50.8840, 5.9810),   # Heerlerbaan 96
    ('employees 12', 50.8860, 5.9835),   # Jeruzalemstraat 16
    ('employees 13', 50.8850, 5.9880),   # Caumerboord 86
    ('employees 14', 50.8890, 5.9822),   # Pattonstraat 11
    ('employees 15', 50.8710, 5.9920),   # Eisterweg 4
    ('employees 16', 50.8810, 5.9680),   # Lindelaan 43
    ('employees 17', 50.8952, 5.9672),   # Mariabad 8
    ('employees 18', 50.8875, 5.9805),   # Samariastraat 7
    ('employees 19', 50.8940, 5.9665),   # Ds. Jongeneelstraat 7
    ('employees 20', 50.8790, 5.9760),   # Welterlaan 23
]

employees_df = pd.DataFrame(employee_data, columns=['name', 'lat', 'lon'])
print(f'Loaded {len(employees_df)} employees.')
print(employees_df)

## 4. Bestemmingspunten definiëren

Vijf bestemmingspunten worden verspreid over Heerlen geplaatst om een goede geografische spreiding te garanderen.

In [ ]:
# Five destination points spread across Heerlen
destinations = [
    {'id': 'D1', 'name': 'Centrum / Maankwartier',  'lat': 50.8882, 'lon': 5.9800},
    {'id': 'D2', 'name': 'Heerlen-Noord / Passart', 'lat': 50.9020, 'lon': 5.9780},
    {'id': 'D3', 'name': 'MSP Campus / Hoensbroek',  'lat': 50.9060, 'lon': 5.9300},
    {'id': 'D4', 'name': 'Heerlerbaan / Zuid',       'lat': 50.8720, 'lon': 5.9850},
    {'id': 'D5', 'name': 'Woonboulevard Heerlen',    'lat': 50.8835, 'lon': 5.9560},
]
destinations_df = pd.DataFrame(destinations)
print('Destination points:')
print(destinations_df)

## 5. Koppelen aan graph-knopen

Elke medewerker en elk bestemmingspunt wordt gekoppeld aan het dichtstbijzijnde knoop in het wegennet.

In [ ]:
def nearest_node(lon, lat):
    """Find the nearest graph node to the given coordinates."""
    dist, idx = tree.query([lon, lat])
    return node_ids[idx], dist

# Snap employees to nearest graph node
employees_df['node'], employees_df['snap_dist'] = zip(
    *employees_df.apply(lambda r: nearest_node(r['lon'], r['lat']), axis=1)
)

# Snap destinations to nearest graph node
destinations_df['node'], destinations_df['snap_dist'] = zip(
    *destinations_df.apply(lambda r: nearest_node(r['lon'], r['lat']), axis=1)
)

print('Employee → nearest node mapping (first 5):')
print(employees_df[['name', 'node', 'snap_dist']].head())
print('\nDestination → nearest node mapping:')
print(destinations_df[['name', 'node', 'snap_dist']])

## 6. Optimale bestemmingstoewijzing

Voor elke medewerker berekenen we de reistijd naar elk van de 5 bestemmingen. De medewerker wordt toegewezen aan de bestemming waarnaar de kortste reistijd bestaat (Dijkstra).

In [ ]:
dest_nodes = destinations_df['node'].tolist()
dest_ids   = destinations_df['id'].tolist()

# Pre-compute shortest paths FROM each destination node (reverse = from dest to all)
# This is more efficient: one Dijkstra per destination instead of one per employee
dist_from_dest = {}
for dest_node, dest_id in zip(dest_nodes, dest_ids):
    lengths = nx.single_source_dijkstra_path_length(G, dest_node, weight='weight')
    dist_from_dest[dest_id] = lengths
    print(f'Shortest paths from {dest_id} computed.')

# Assign each employee to the nearest destination
def assign_destination(emp_node):
    best_dest  = None
    best_time  = float('inf')
    for dest_id, lengths in dist_from_dest.items():
        t = lengths.get(emp_node, float('inf'))
        if t < best_time:
            best_time = t
            best_dest = dest_id
    return best_dest, best_time

employees_df['assigned_dest'], employees_df['travel_time_min'] = zip(
    *employees_df['node'].apply(assign_destination)
)

print('\nEmployee assignments:')
print(employees_df[['name', 'assigned_dest', 'travel_time_min']].to_string())

## 7. Routeberekening per medewerker

Voor elke medewerker berekenen we de exacte padsequentie (lijst van knopen) via Dijkstra.

In [ ]:
# Determine the destination node for each employee
dest_node_map = dict(zip(destinations_df['id'], destinations_df['node']))

routes = []  # list of (employee_idx, path_nodes)

for idx, emp in employees_df.iterrows():
    src_node  = emp['node']
    dest_node = dest_node_map[emp['assigned_dest']]
    try:
        path = nx.shortest_path(G, source=src_node, target=dest_node, weight='weight')
        routes.append({'emp_idx': idx, 'path': path, 'success': True})
    except nx.NetworkXNoPath:
        routes.append({'emp_idx': idx, 'path': [], 'success': False})
        print(f'No path found for {emp["name"]} → {emp["assigned_dest"]}')

successful = sum(1 for r in routes if r['success'])
print(f'Routes computed: {successful}/{len(routes)} successful.')

## 8. Kaartvisualisatie

Elke medewerker krijgt een unieke kleur. We tekenen:
- Het wegennet als lichtgrijs achtergrond
- De route van elke medewerker naar zijn/haar bestemming
- Medewerkerlocaties als gekleurde markers
- Bestemmingspunten als grote zwarte sterren

In [ ]:
# 20 visually distinct colors for employees
EMPLOYEE_COLORS = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4',
    '#469990', '#dcbeff', '#9A6324', '#fffac8', '#800000',
    '#aaffc3', '#808000', '#ffd8b1', '#000075', '#a9a9a9',
]

# Destination marker colors
DEST_COLORS = {
    'D1': '#000000',
    'D2': '#1a1a2e',
    'D3': '#16213e',
    'D4': '#0f3460',
    'D5': '#533483',
}

# Build edge geometry lookup for fast drawing
edge_geom = {}
for _, row in edges_df.iterrows():
    u, v = row['u'], row['v']
    edge_geom[(u, v)] = row['geometry']
    edge_geom[(v, u)] = row['geometry']  # undirected

def path_latlon(path_nodes):
    """Convert a node path to a list of (lat, lon) coordinate segments."""
    segments = []
    for i in range(len(path_nodes) - 1):
        u, v = path_nodes[i], path_nodes[i + 1]
        geom = edge_geom.get((u, v))
        if geom is not None:
            coords = list(geom.coords)
            segments.extend([(lat, lon) for lon, lat in coords])
        else:
            # Fallback: straight line
            cu = node_coords.get(u)
            cv = node_coords.get(v)
            if cu and cv:
                segments += [(cu[1], cu[0]), (cv[1], cv[0])]
    return segments

# Compute map center
center_lat = np.mean(node_lats)
center_lon = np.mean(node_lons)

# --- Build map ---
m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')

# Draw road network as light gray background
for _, row in edges_df.iterrows():
    coords = list(row['geometry'].coords)
    latlon = [(lat, lon) for lon, lat in coords]
    folium.PolyLine(locations=latlon, color='#cccccc', weight=1, opacity=0.4).add_to(m)

# Draw routes
for route in routes:
    if not route['success'] or not route['path']:
        continue
    idx   = route['emp_idx']
    color = EMPLOYEE_COLORS[idx % len(EMPLOYEE_COLORS)]
    emp   = employees_df.loc[idx]
    latlon = path_latlon(route['path'])
    if latlon:
        folium.PolyLine(
            locations=latlon,
            color=color,
            weight=4,
            opacity=0.85,
            tooltip=f"{emp['name']} → {emp['assigned_dest']} ({emp['travel_time_min']:.1f} min)"
        ).add_to(m)

# Draw employee start markers
for idx, emp in employees_df.iterrows():
    color = EMPLOYEE_COLORS[idx % len(EMPLOYEE_COLORS)]
    folium.CircleMarker(
        location=[emp['lat'], emp['lon']],
        radius=7,
        color='white',
        weight=2,
        fill=True,
        fill_color=color,
        fill_opacity=1.0,
        popup=folium.Popup(
            f"<b>{emp['name']}</b><br>"
            f"Bestemming: {emp['assigned_dest']}<br>"
            f"Reistijd: {emp['travel_time_min']:.1f} min",
            max_width=200
        ),
        tooltip=emp['name']
    ).add_to(m)

# Draw destination markers (large star icons)
for _, dest in destinations_df.iterrows():
    folium.Marker(
        location=[dest['lat'], dest['lon']],
        icon=folium.Icon(color='black', icon='star', prefix='fa'),
        popup=folium.Popup(
            f"<b>{dest['id']}: {dest['name']}</b>",
            max_width=200
        ),
        tooltip=f"{dest['id']}: {dest['name']}"
    ).add_to(m)

print('Map built successfully.')

## 9. Legenda toevoegen en kaart opslaan

In [ ]:
# Build legend HTML
legend_rows = ''
for idx, emp in employees_df.iterrows():
    color = EMPLOYEE_COLORS[idx % len(EMPLOYEE_COLORS)]
    dest_label = emp['assigned_dest']
    row_html = (
        '<tr>'
        f'<td><div style="width:16px;height:16px;background:{color};'
        'border:1px solid #fff;border-radius:3px;"></div></td>'
        f'<td style="padding-left:6px;">{emp["name"]}</td>'
        f'<td style="padding-left:8px;color:#555;">{dest_label} · {emp["travel_time_min"]:.1f} min</td>'
        '</tr>'
    )
    legend_rows += row_html

dest_legend = ''.join(
    '<tr><td><span style="font-size:16px;">★</span></td>'
    f'<td colspan="2" style="padding-left:6px;"><b>{d["id"]}</b>: {d["name"]}</td></tr>'
    for _, d in destinations_df.iterrows()
)

legend_html = (
    '<div style="position:fixed;bottom:30px;left:30px;z-index:1000;'
    'background:white;padding:10px 14px;border-radius:6px;'
    'font-size:11px;font-family:sans-serif;'
    'box-shadow:0 2px 8px rgba(0,0,0,0.25);max-height:480px;overflow-y:auto;">'
    '<b style="font-size:13px;">Legenda</b>'
    '<table style="border-collapse:collapse;margin-top:6px;">'
    '<tr><td colspan="3" style="font-weight:bold;padding-bottom:4px;">Medewerkers</td></tr>'
    + legend_rows +
    '<tr><td colspan="3" style="font-weight:bold;padding-top:8px;padding-bottom:4px;">Bestemmingen</td></tr>'
    + dest_legend +
    '</table></div>'
)

m.get_root().html.add_child(folium.Element(legend_html))

# Save the map
output_path = '../output/employee_routes_map.html'
m.save(output_path)
print(f'Map saved to {output_path}')

# Display inline in notebook
from IPython.display import IFrame, display
display(IFrame(src='employee_routes_map.html', width='100%', height=650))


## 10. Statistieken per bestemming

In [ ]:
print('=== Samenvatting per bestemming ===')
summary = employees_df.groupby('assigned_dest').agg(
    aantal_medewerkers=('name', 'count'),
    gem_reistijd_min=('travel_time_min', 'mean'),
    min_reistijd_min=('travel_time_min', 'min'),
    max_reistijd_min=('travel_time_min', 'max'),
).round(2)

# Add destination name
dest_name_map = dict(zip(destinations_df['id'], destinations_df['name']))
summary['naam'] = summary.index.map(dest_name_map)
print(summary[['naam', 'aantal_medewerkers', 'gem_reistijd_min', 'min_reistijd_min', 'max_reistijd_min']].to_string())

print(f'\nTotale gemiddelde reistijd: {employees_df["travel_time_min"].mean():.2f} minuten')
print(f'Kortste route: {employees_df["travel_time_min"].min():.2f} min ({employees_df.loc[employees_df["travel_time_min"].idxmin(), "name"]})')
print(f'Langste route: {employees_df["travel_time_min"].max():.2f} min ({employees_df.loc[employees_df["travel_time_min"].idxmax(), "name"]})')